# 資料處理

## 讀nc4

In [12]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm


def check_data_folder(folder: str) -> bool:
    return os.path.exists(folder) and os.path.isdir(folder)


def load_data(file_path: str) -> xr.Dataset:
    """
    Load data from a NetCDF file, trying netcdf4 then h5netcdf.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    # 優先 netcdf4
    try:
        return xr.open_dataset(file_path, engine="netcdf4")
    except Exception as e1:
        # 改試 h5netcdf
        try:
            return xr.open_dataset(file_path, engine="h5netcdf")
        except Exception as e2:
            raise RuntimeError(
                f"Failed to open {file_path} with netcdf4 and h5netcdf.\n"
                f"e1: {e1}\n"
                f"e2: {e2}\n"
                "請確認這個環境有安裝 netCDF4 或 h5netcdf。"
            )


# ================= main program =================

data_folder = "nc4"
var_name = "T2M"  # 目標變數名稱（請確認檔內真的叫這個）

# 1. 檢查資料夾
if not check_data_folder(data_folder):
    raise FileNotFoundError(f"Data folder not found: {data_folder}")
print(f"Data folder found: {data_folder}")

# 2. 找出所有像 1980-01.nc4 的檔案
pattern = os.path.join(data_folder, "*.nc4")
file_list = sorted(glob.glob(pattern))

if len(file_list) == 0:
    raise FileNotFoundError(f"No nc4 files found with pattern: {pattern}")

print(f"Found {len(file_list)} files.")
print("First 5 files:", file_list[:5])

# 3. 用第一個檔案確認經緯度與變數存在，必要時自動偵測 var_name
with load_data(file_list[0]) as sample_data:
    print("Data variables in first file:")
    print(list(sample_data.data_vars))
    print("Coordinates:")
    print({k: sample_data[k].shape for k in sample_data.coords})

    # 找出所有長得像 (time, lat, lon) 的候選變數
    candidates = []
    for v in sample_data.data_vars:
        dims = set(sample_data[v].dims)
        if {"time", "lat", "lon"}.issubset(dims):
            candidates.append(v)

    # 如果原本設定的 var_name 不在，就試著自動改
    if var_name not in sample_data.data_vars:
        if len(candidates) == 1:
            auto_var = candidates[0]
            print(f"[Info] Variable '{var_name}' not found, auto-select '{auto_var}' as target.")
            var_name = auto_var
        else:
            raise KeyError(
                f"Variable '{var_name}' not found in file: {file_list[0]}\n"
                f"Available data_vars: {list(sample_data.data_vars)}\n"
                f"3D (time,lat,lon) candidates: {candidates}\n"
                "請將上方其中一個正確變數名稱填入 var_name。"
            )

    # 取經緯度
    if "lat" not in sample_data.coords or "lon" not in sample_data.coords:
        raise KeyError("lat/lon coordinates not found in sample file.")
    lat = sample_data["lat"].values
    lon = sample_data["lon"].values

nlat = lat.shape[0]
nlon = lon.shape[0]
print(f"Confirmed var_name = {var_name}")
print(f"Lat: {nlat}, Lon: {nlon}")


# 4. 逐檔讀入，累積到 list
data_list = []
time_list = []

for f in tqdm(file_list, desc="Combining"):
    with load_data(f) as ds:
        da = ds[var_name]  # (time, lat, lon)

        # 確保 lat/lon 一致（保險，可視情況註解）
        if da.sizes["lat"] != nlat or da.sizes["lon"] != nlon:
            raise ValueError(f"Lat/Lon size mismatch in file: {f}")

        # 資料轉 float32，省記憶體
        data_list.append(da.values.astype(np.float32))

        if "time" not in ds:
            raise KeyError(f"'time' coordinate not found in file: {f}")

        # decode_cf 確保時間是真正 datetime
        t = xr.decode_cf(ds)["time"].values
        time_list.append(t.astype("datetime64[ns]"))

# 5. 串起來 → (ntot, lat, lon) & DatetimeIndex
combined = np.concatenate(data_list, axis=0)   # (ntot, nlat, nlon)
time_array = np.concatenate(time_list, axis=0) # (ntot,)

if combined.shape[0] != time_array.shape[0]:
    raise ValueError(
        f"time length ({time_array.shape[0]}) "
        f"!= data length ({combined.shape[0]})"
    )

# 依時間排序（通常已排序，這裡是保險）
sort_idx = np.argsort(time_array)
combined = combined[sort_idx]
time_array = time_array[sort_idx]

time_index = pd.to_datetime(time_array)

print(f"Combined data shape: {combined.shape}")
print(f"Time index: {time_index[0]} -> {time_index[-1]} (len={len(time_index)})")

# 6. 攤平成 cell × time
ntot, nlat, nlon = combined.shape
ncell = nlat * nlon

# (cell, time)
y_all = combined.reshape(ntot, ncell).T

# 建立每個 cell 的 (lon, lat)
lon_grid, lat_grid = np.meshgrid(lon, lat)
gg = np.column_stack([lon_grid.ravel(), lat_grid.ravel()])  # (cell, 2)

print(f"y_all shape: {y_all.shape}  (cells x time)")
print(f"gg shape: {gg.shape}        (cells x [lon, lat])")


Data folder found: nc4
Found 548 files.
First 5 files: ['nc4/1980-01.nc4', 'nc4/1980-02.nc4', 'nc4/1980-03.nc4', 'nc4/1980-04.nc4', 'nc4/1980-05.nc4']
Data variables in first file:
['T2M', 'T2MDEW', 'Var_T2M', 'T2MWET']
Coordinates:
{'lon': (576,), 'time': (1,), 'lat': (361,)}
Confirmed var_name = T2M
Lat: 361, Lon: 576


Combining: 100%|██████████| 548/548 [00:06<00:00, 90.94it/s] 


Combined data shape: (548, 361, 576)
Time index: 1980-01-01 00:30:00 -> 2025-08-01 00:30:00 (len=548)
y_all shape: (207936, 548)  (cells x time)
gg shape: (207936, 2)        (cells x [lon, lat])


# 限制範圍抽樣 in USA

## 500個格點

In [13]:
# ===== 只看美國本土範圍，並從中抽樣 500 個格點 =====
import numpy as np

# gg: (ncell, 2)；第 0 欄是 lon，第 1 欄是 lat
lon_all = gg[:, 0].astype(float)
lat_all = gg[:, 1].astype(float)

# 如果經度是 0~360，轉成 -180~180
lon_all_180 = ((lon_all + 180) % 360) - 180

# 美國本土 48 州的大致範圍
lon_min, lon_max = -125, -66
lat_min, lat_max = 24, 50

# 建立遮罩：只保留在美國本土範圍內的格點
mask_us_mainland = (
    (lon_all_180 >= lon_min) & (lon_all_180 <= lon_max) &
    (lat_all      >= lat_min) & (lat_all      <= lat_max)
)

idx_us_mainland = np.where(mask_us_mainland)[0]
print(f"美國本土範圍內的格點數量: {len(idx_us_mainland)}")

if len(idx_us_mainland) == 0:
    raise ValueError("在設定的美國本土範圍內沒有格點，請調整 lon_min/max 或 lat_min/max。")

# 只保留美國本土子集合
y_us = y_all[idx_us_mainland, :]  # (n_us, ntot)
coords_us = np.column_stack([
    lon_all_180[idx_us_mainland],
    lat_all[idx_us_mainland]
])

print(f"美國本土子集合 y_us shape: {y_us.shape}")
print(f"美國本土子集合 coords_us shape: {coords_us.shape}")

# ===== 抽樣 500 個格點 =====
N_SAMPLE_TARGET = 500
n_sample = min(N_SAMPLE_TARGET, len(idx_us_mainland))

np.random.seed(42)  # 為了可重現
sample_idx_in_us = np.random.choice(
    len(idx_us_mainland),
    size=n_sample,
    replace=False
)

# 原始 global index
sample_idx_global = idx_us_mainland[sample_idx_in_us]

# 抽樣後資料
y_sample = y_all[sample_idx_global, :]  # (n_sample, ntot)
coords_sample = np.column_stack([
    lon_all_180[sample_idx_global],
    lat_all[sample_idx_global]
])

lon_us_sample = coords_sample[:, 0]
lat_us_sample = coords_sample[:, 1]

print(f"抽樣 {n_sample} 個美國本土格點")
print("前 5 個 sample index (global):", sample_idx_global[:5])
print("前 5 個 sample 座標 (lon, lat):")
for i in range(min(5, n_sample)):
    print(f"  #{i}: lon={lon_us_sample[i]:.2f}, lat={lat_us_sample[i]:.2f}")

美國本土範圍內的格點數量: 5035
美國本土子集合 y_us shape: (5035, 548)
美國本土子集合 coords_us shape: (5035, 2)
抽樣 500 個美國本土格點
前 5 個 sample index (global): [153335 159677 145263 156825 138939]
前 5 個 sample 座標 (lon, lat):
  #0: lon=-105.62, lat=43.00
  #1: lon=-101.88, lat=48.50
  #2: lon=-110.62, lat=36.00
  #3: lon=-84.38, lat=46.00
  #4: lon=-103.12, lat=30.50


### DLinear

#### tune par (grid search)

In [ ]:
import json
import time
import gc
import os
import logging
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

from darts import TimeSeries
from darts.models import DLinearModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


print("\n===== DLinear hyperparameter search with Optuna (train/val/test = 70/15/15) =====")

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

# Global settings & search controls
RUN_SEARCH = False  # 是否執行搜尋；如果 False，會嘗試從 BEST_RESULT_PATH 載入結果並顯示（如果檔案存在的話）
N_TRIALS = 500
TIMEOUT_SECONDS = None

USE_COVARIATES = False
USE_REVIN = True
RANDOM_STATE = 42

PL_TRAINER_KWARGS = {
    "enable_progress_bar": False,
    "logger": False,
    "enable_checkpointing": False,
    "enable_model_summary": False,
}

SAVE_DIR = Path(".")
BEST_RESULT_PATH = SAVE_DIR / "best_dlinear_result.json"
BEST_PARAMS_PATH = SAVE_DIR / "best_dlinear_params.json"


def _json_dump(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def _json_load(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _cleanup_torch_cache() -> None:
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def _ts_to_2d(ts: TimeSeries) -> np.ndarray:
    arr = np.asarray(ts.all_values(copy=False))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr


def _build_model_kwargs(
    in_len: int,
    out_len: int,
    ksize: int,
    n_epochs: int,
    bs: int,
    lr: float,
    wd: float,
    const_init: bool,
) -> dict:
    return {
        "input_chunk_length": int(in_len),
        "output_chunk_length": int(out_len),
        "kernel_size": int(ksize),
        "n_epochs": int(n_epochs),
        "random_state": int(RANDOM_STATE),
        "use_reversible_instance_norm": bool(USE_REVIN),
        "batch_size": int(bs),
        "optimizer_kwargs": {
            "lr": float(lr),
            "weight_decay": float(wd),
        },
        "const_init": bool(const_init),
        "pl_trainer_kwargs": PL_TRAINER_KWARGS,
        "log_tensorboard": False,
        "save_checkpoints": False,
    }


def _compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    r2 = r2_score(y_true, y_pred)
    return {
        "rmse": float(rmse),
        "mse": float(mse),
        "mae": float(mae),
        "r2": float(r2),
    }


def _silent_call(func, *args, **kwargs):
    with open(os.devnull, "w") as fnull:
        with contextlib.redirect_stdout(fnull), contextlib.redirect_stderr(fnull):
            return func(*args, **kwargs)


SAMPLE_SIZE = int(n_sample)
T = int(y_sample.shape[1])

if y_sample.ndim != 2:
    raise ValueError(f"y_sample 必須是 2 維陣列，目前 shape = {y_sample.shape}")

if y_sample.shape[0] != SAMPLE_SIZE:
    raise ValueError(
        f"n_sample = {SAMPLE_SIZE}，但 y_sample.shape[0] = {y_sample.shape[0]}，兩者不一致"
    )

if len(time_index) != T:
    raise ValueError(
        f"time_index 長度 = {len(time_index)}，但 y_sample.shape[1] = {T}，兩者不一致"
    )

ts_df = pd.DataFrame(
    y_sample.T,
    index=time_index,
    columns=[f"cell_{i}" for i in range(SAMPLE_SIZE)]
).astype("float32")

print("Total time steps:", T)
print("SAMPLE_SIZE:", SAMPLE_SIZE)
print("ts_df shape:", ts_df.shape)

month_values = time_index.month.astype("float32")
month_df = pd.DataFrame({"month": month_values}, index=time_index)
month_ts = TimeSeries.from_dataframe(month_df.astype("float32"))

train_frac = 0.7
val_frac = 0.15
test_frac = 0.15

if not np.isclose(train_frac + val_frac + test_frac, 1.0):
    raise ValueError("train_frac + val_frac + test_frac 必須等於 1")

cut_train = int(T * train_frac)
cut_val = int(T * (train_frac + val_frac))

idx = ts_df.index
split_time_train = idx[cut_train]
split_time_val = idx[cut_val]

ts_all_raw = TimeSeries.from_dataframe(ts_df)
trainval_raw, test_raw = ts_all_raw.split_before(split_time_val)
train_raw, val_raw = trainval_raw.split_before(split_time_train)

print(f"Train len (raw): {len(train_raw)}, Val len: {len(val_raw)}, Test len: {len(test_raw)}")

month_trainval, month_test = month_ts.split_before(split_time_val)
month_train, month_val = month_trainval.split_before(split_time_train)

assert len(month_train) == len(train_raw)
assert len(month_val) == len(val_raw)
assert len(month_test) == len(test_raw)

train_df = ts_df.iloc[:cut_train]
mean_vec = train_df.mean(axis=0)
std_vec = train_df.std(axis=0).replace(0.0, 1.0)

ts_df_scaled = (ts_df - mean_vec) / std_vec
ts_all_scaled = TimeSeries.from_dataframe(ts_df_scaled)

trainval_ts, test_ts = ts_all_scaled.split_before(split_time_val)
train_ts, val_ts = trainval_ts.split_before(split_time_train)

T_train, T_val, T_test = len(train_ts), len(val_ts), len(test_ts)
print("\n[Scaled lengths] T_train =", T_train, "T_val =", T_val, "T_test =", T_test)

val_true_raw = ts_df.to_numpy(dtype=np.float32)[cut_train:cut_val, :]
test_true_raw = ts_df.to_numpy(dtype=np.float32)[cut_val:, :]


def inverse_scale(x: np.ndarray) -> np.ndarray:
    return x * std_vec.values + mean_vec.values


SEARCH_SPACE = {
    "INPUT_CHUNK_LENGTH": [24, 36, 48],
    "OUTPUT_CHUNK_LENGTH": [12, 24],
    "KERNEL_SIZE": [15, 25],
    "N_EPOCHS": [30, 50],
    "BATCH_SIZE": [16, 32, 64],
    "LR": [1e-4, 2e-4, 3e-4],
    "WEIGHT_DECAY": [0.0, 1e-5],
    "CONST_INIT": [True, False],
}

keys = list(SEARCH_SPACE.keys())

if (not RUN_SEARCH) and BEST_RESULT_PATH.exists():
    best_loaded = _json_load(BEST_RESULT_PATH)

    print("\n===== LOADED BEST RESULT (skip search) =====")
    print("Best VAL metrics:")
    print(f"  RMSE = {best_loaded['rmse_val']:.6f}")
    print(f"  MSE  = {best_loaded['mse_val']:.6f}")
    print(f"  MAE  = {best_loaded['mae_val']:.6f}")
    print(f"  R2   = {best_loaded['r2_val']:.6f}")

    print("\nBest TEST metrics:")
    print(f"  RMSE = {best_loaded['rmse_test']:.6f}")
    print(f"  MSE  = {best_loaded['mse_test']:.6f}")
    print(f"  MAE  = {best_loaded['mae_test']:.6f}")
    print(f"  R2   = {best_loaded['r2_test']:.6f}")

    print("\nBest params:")
    for k in keys:
        print(f"  {k} = {best_loaded['params'][k]}")
    print(f"  USE_REVIN = {best_loaded['params']['USE_REVIN']}")
    print(f"  USE_COVARIATES = {best_loaded['params']['USE_COVARIATES']}")
    print(f"  RANDOM_STATE = {best_loaded['params']['RANDOM_STATE']}")
else:
    def objective(trial: optuna.Trial) -> float:
        print(f"[Trial {trial.number + 1}/{N_TRIALS}]", flush=True)

        in_len = trial.suggest_categorical(
            "INPUT_CHUNK_LENGTH",
            SEARCH_SPACE["INPUT_CHUNK_LENGTH"]
        )
        out_len = trial.suggest_categorical(
            "OUTPUT_CHUNK_LENGTH",
            SEARCH_SPACE["OUTPUT_CHUNK_LENGTH"]
        )
        ksize = trial.suggest_categorical(
            "KERNEL_SIZE",
            SEARCH_SPACE["KERNEL_SIZE"]
        )
        n_epochs = trial.suggest_categorical(
            "N_EPOCHS",
            SEARCH_SPACE["N_EPOCHS"]
        )
        bs = trial.suggest_categorical(
            "BATCH_SIZE",
            SEARCH_SPACE["BATCH_SIZE"]
        )
        lr = trial.suggest_categorical(
            "LR",
            SEARCH_SPACE["LR"]
        )
        wd = trial.suggest_categorical(
            "WEIGHT_DECAY",
            SEARCH_SPACE["WEIGHT_DECAY"]
        )
        const_init = trial.suggest_categorical(
            "CONST_INIT",
            SEARCH_SPACE["CONST_INIT"]
        )

        model_kwargs = _build_model_kwargs(
            in_len=in_len,
            out_len=out_len,
            ksize=ksize,
            n_epochs=n_epochs,
            bs=bs,
            lr=lr,
            wd=wd,
            const_init=const_init,
        )

        model = None
        pred_val = None

        fit_kwargs = {
            "series": train_ts,
            "verbose": False,
        }
        pred_val_kwargs = {
            "n": T_val,
            "verbose": False,
            "show_warnings": False,
        }

        if USE_COVARIATES:
            fit_kwargs["past_covariates"] = month_train
            fit_kwargs["future_covariates"] = month_train
            pred_val_kwargs["past_covariates"] = month_trainval
            pred_val_kwargs["future_covariates"] = month_trainval

        try:
            model = DLinearModel(**model_kwargs)
            _silent_call(model.fit, **fit_kwargs)
            pred_val = _silent_call(model.predict, **pred_val_kwargs)

            pred_val_scaled = _ts_to_2d(pred_val)
            pred_val_raw = inverse_scale(pred_val_scaled)

            y_val_true = val_true_raw.reshape(-1, SAMPLE_SIZE)
            y_val_pred = pred_val_raw.reshape(-1, SAMPLE_SIZE)

            val_metrics = _compute_metrics(y_val_true, y_val_pred)

            trial.set_user_attr("rmse_val", val_metrics["rmse"])
            trial.set_user_attr("mse_val", val_metrics["mse"])
            trial.set_user_attr("mae_val", val_metrics["mae"])
            trial.set_user_attr("r2_val", val_metrics["r2"])
            trial.set_user_attr("USE_REVIN", bool(USE_REVIN))
            trial.set_user_attr("USE_COVARIATES", bool(USE_COVARIATES))
            trial.set_user_attr("RANDOM_STATE", int(RANDOM_STATE))

            return trial.user_attrs["rmse_val"]

        except Exception:
            raise optuna.TrialPruned()

        finally:
            del model
            del pred_val
            gc.collect()
            _cleanup_torch_cache()

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=0),
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        timeout=TIMEOUT_SECONDS,
        gc_after_trial=True,
        show_progress_bar=False,
    )

    print("\n===== BEST PARAMS (by VAL RMSE) =====")

    successful_trials = [
        t for t in study.trials
        if t.state == optuna.trial.TrialState.COMPLETE
    ]

    print(f"Successful trials: {len(successful_trials)} / {len(study.trials)}")

    if len(successful_trials) == 0:
        print("No successful run in the search space.")
    else:
        best_trial = study.best_trial

        best_params = {
            "INPUT_CHUNK_LENGTH": best_trial.params["INPUT_CHUNK_LENGTH"],
            "OUTPUT_CHUNK_LENGTH": best_trial.params["OUTPUT_CHUNK_LENGTH"],
            "KERNEL_SIZE": best_trial.params["KERNEL_SIZE"],
            "N_EPOCHS": best_trial.params["N_EPOCHS"],
            "BATCH_SIZE": best_trial.params["BATCH_SIZE"],
            "LR": float(best_trial.params["LR"]),
            "WEIGHT_DECAY": float(best_trial.params["WEIGHT_DECAY"]),
            "CONST_INIT": bool(best_trial.params["CONST_INIT"]),
            "USE_REVIN": bool(USE_REVIN),
            "USE_COVARIATES": bool(USE_COVARIATES),
            "RANDOM_STATE": int(RANDOM_STATE),
        }

        print("Best VAL metrics:")
        print(f"  RMSE = {best_trial.user_attrs['rmse_val']:.6f}")
        print(f"  MSE  = {best_trial.user_attrs['mse_val']:.6f}")
        print(f"  MAE  = {best_trial.user_attrs['mae_val']:.6f}")
        print(f"  R2   = {best_trial.user_attrs['r2_val']:.6f}")

        best_model_kwargs = _build_model_kwargs(
            in_len=best_params["INPUT_CHUNK_LENGTH"],
            out_len=best_params["OUTPUT_CHUNK_LENGTH"],
            ksize=best_params["KERNEL_SIZE"],
            n_epochs=best_params["N_EPOCHS"],
            bs=best_params["BATCH_SIZE"],
            lr=best_params["LR"],
            wd=best_params["WEIGHT_DECAY"],
            const_init=best_params["CONST_INIT"],
        )

        best_model = DLinearModel(**best_model_kwargs)

        fit_best_kwargs = {
            "series": trainval_ts,
            "verbose": False,
        }
        pred_test_kwargs = {
            "n": T_test,
            "verbose": False,
            "show_warnings": False,
        }

        if USE_COVARIATES:
            fit_best_kwargs["past_covariates"] = month_trainval
            fit_best_kwargs["future_covariates"] = month_trainval
            pred_test_kwargs["past_covariates"] = month_ts
            pred_test_kwargs["future_covariates"] = month_ts

        _silent_call(best_model.fit, **fit_best_kwargs)
        pred_test = _silent_call(best_model.predict, **pred_test_kwargs)

        pred_test_scaled = _ts_to_2d(pred_test)
        pred_test_raw = inverse_scale(pred_test_scaled)

        y_test_true = test_true_raw.reshape(-1, SAMPLE_SIZE)
        y_test_pred = pred_test_raw.reshape(-1, SAMPLE_SIZE)

        test_metrics = _compute_metrics(y_test_true, y_test_pred)

        best_result_payload = {
            "rmse_val": float(best_trial.user_attrs["rmse_val"]),
            "mse_val": float(best_trial.user_attrs["mse_val"]),
            "mae_val": float(best_trial.user_attrs["mae_val"]),
            "r2_val": float(best_trial.user_attrs["r2_val"]),
            "rmse_test": float(test_metrics["rmse"]),
            "mse_test": float(test_metrics["mse"]),
            "mae_test": float(test_metrics["mae"]),
            "r2_test": float(test_metrics["r2"]),
            "params": best_params,
            "search_space": SEARCH_SPACE,
            "n_trials": int(N_TRIALS),
            "best_trial_number": int(best_trial.number),
            "successful_trials": int(len(successful_trials)),
            "split_ratio": {
                "train": train_frac,
                "val": val_frac,
                "test": test_frac,
            },
            "sample_size": int(SAMPLE_SIZE),
            "time_steps": int(T),
        }

        print("\nBest TEST metrics:")
        print(f"  RMSE = {best_result_payload['rmse_test']:.6f}")
        print(f"  MSE  = {best_result_payload['mse_test']:.6f}")
        print(f"  MAE  = {best_result_payload['mae_test']:.6f}")
        print(f"  R2   = {best_result_payload['r2_test']:.6f}")

        print("\nBest params:")
        for k in keys:
            print(f"  {k} = {best_params[k]}")
        print(f"  USE_REVIN = {best_params['USE_REVIN']}")
        print(f"  USE_COVARIATES = {best_params['USE_COVARIATES']}")
        print(f"  RANDOM_STATE = {best_params['RANDOM_STATE']}")
        print(f"  BEST_TRIAL_NUMBER = {best_trial.number}")

        _json_dump(best_result_payload, BEST_RESULT_PATH)
        _json_dump(best_params, BEST_PARAMS_PATH)

        print("\nSaved:")
        print(f"  - {BEST_RESULT_PATH.resolve()}")
        print(f"  - {BEST_PARAMS_PATH.resolve()}")

        del best_model
        del pred_test, pred_test_scaled, pred_test_raw
        del y_test_true, y_test_pred, test_metrics
        gc.collect()
        _cleanup_torch_cache()


===== DLinear hyperparameter search with Optuna (train/val/test = 70/15/15) =====
Total time steps: 548
SAMPLE_SIZE: 500
ts_df shape: (548, 500)
Train len (raw): 383, Val len: 82, Test len: 83

[Scaled lengths] T_train = 383 T_val = 82 T_test = 83
[Trial 1/500]
[Trial 2/500]
[Trial 3/500]
[Trial 4/500]
[Trial 5/500]
[Trial 6/500]
[Trial 7/500]
[Trial 8/500]
[Trial 9/500]
[Trial 10/500]
[Trial 11/500]
[Trial 12/500]
[Trial 13/500]
[Trial 14/500]
[Trial 15/500]
[Trial 16/500]
[Trial 17/500]
[Trial 18/500]
[Trial 19/500]
[Trial 20/500]
[Trial 21/500]
[Trial 22/500]
[Trial 23/500]
[Trial 24/500]
[Trial 25/500]
[Trial 26/500]
[Trial 27/500]
[Trial 28/500]
[Trial 29/500]
[Trial 30/500]
[Trial 31/500]
[Trial 32/500]
[Trial 33/500]
[Trial 34/500]
[Trial 35/500]
[Trial 36/500]
[Trial 37/500]
[Trial 38/500]
[Trial 39/500]
[Trial 40/500]
[Trial 41/500]
[Trial 42/500]
[Trial 43/500]
[Trial 44/500]
[Trial 45/500]
[Trial 46/500]
[Trial 47/500]
[Trial 48/500]
[Trial 49/500]
[Trial 50/500]
[Trial 51/

#### formal

In [15]:
# ============================================================
# 8) Re-run DLinear with BEST params (train+val -> test)
#    Priority:
#      (1) best["params"] in memory
#      (2) load from best_dlinear_params.json
# ============================================================
from pathlib import Path
import json
import gc

BEST_PARAMS_PATH = Path("best_dlinear_params.json")
RERUN_METRICS_PATH = Path("best_dlinear_rerun_metrics.json")

def _json_load(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _json_dump(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

bp = None

if "best" in globals() and isinstance(best, dict) and (best.get("params") is not None):
    bp = best["params"]
elif BEST_PARAMS_PATH.exists():
    bp = _json_load(BEST_PARAMS_PATH)

if bp is None:
    raise RuntimeError(
        "No best params found. Either run search to populate best['params'], "
        "or make sure best_dlinear_params.json exists."
    )

print("\n===== RE-RUN DLinear with BEST params (train+val -> test, split = 70/15/15) =====")
print("Best params used:")
for k in keys:
    if k in bp:
        print(f"  {k} = {bp[k]}")
print(f"  USE_REVIN = {bp.get('USE_REVIN', USE_REVIN)}")
print(f"  USE_COVARIATES = {bp.get('USE_COVARIATES', USE_COVARIATES)}")
print(f"  RANDOM_STATE = {bp.get('RANDOM_STATE', RANDOM_STATE)}")

model_best_kwargs = dict(
    input_chunk_length=int(bp["INPUT_CHUNK_LENGTH"]),
    output_chunk_length=int(bp["OUTPUT_CHUNK_LENGTH"]),
    kernel_size=int(bp["KERNEL_SIZE"]),
    n_epochs=int(bp["N_EPOCHS"]),
    random_state=int(bp.get("RANDOM_STATE", RANDOM_STATE)),
    use_reversible_instance_norm=bool(bp.get("USE_REVIN", USE_REVIN)),
    batch_size=int(bp["BATCH_SIZE"]),
    optimizer_kwargs={
        "lr": float(bp["LR"]),
        "weight_decay": float(bp["WEIGHT_DECAY"]),
    },
    const_init=bool(bp["CONST_INIT"]),
    pl_trainer_kwargs=PL_TRAINER_KWARGS,
    log_tensorboard=False,
    save_checkpoints=False,
)

_cleanup_torch_cache()
gc.collect()

model_best = DLinearModel(**model_best_kwargs)

fit_kwargs_best = {
    "series": trainval_ts,
    "verbose": False,
}
pred_kwargs_best = {
    "n": T_test,
    "verbose": False,
    "show_warnings": False,
}

if bool(bp.get("USE_COVARIATES", USE_COVARIATES)):
    fit_kwargs_best["past_covariates"] = month_trainval
    fit_kwargs_best["future_covariates"] = month_trainval
    pred_kwargs_best["past_covariates"] = month_ts
    pred_kwargs_best["future_covariates"] = month_ts

model_best.fit(**fit_kwargs_best)
pred_best = model_best.predict(**pred_kwargs_best)

pred_best_scaled = _ts_to_2d(pred_best)
pred_best_raw = inverse_scale(pred_best_scaled)

y_test_true_best = test_true_raw.reshape(-1, SAMPLE_SIZE)
y_test_pred_best = pred_best_raw.reshape(-1, SAMPLE_SIZE)

mse_best = mean_squared_error(y_test_true_best, y_test_pred_best)
mae_best = mean_absolute_error(y_test_true_best, y_test_pred_best)
rmse_best = float(np.sqrt(mse_best))
r2_best = r2_score(y_test_true_best, y_test_pred_best)

metrics_best = pd.DataFrame({
    "Model": ["DLINEAR(best)"],
    "Split": ["TEST"],
    "RMSE": [rmse_best],
    "MSE": [float(mse_best)],
    "MAE": [float(mae_best)],
    "R2": [float(r2_best)],
})

print("\n===== BEST DLINEAR RESULTS (re-run: train+val -> test) =====")
print(metrics_best.to_string(index=False))

rerun_payload = {
    "Model": "DLINEAR(best)",
    "Split": "TEST",
    "RMSE": float(rmse_best),
    "MSE": float(mse_best),
    "MAE": float(mae_best),
    "R2": float(r2_best),
    "params_used": bp,
    "rerun_scheme": "train+val -> test",
    "split_ratio": {
        "train": 0.7,
        "val": 0.15,
        "test": 0.15,
    },
    "sample_size": int(SAMPLE_SIZE),
    "time_steps": int(T),
}
_json_dump(rerun_payload, RERUN_METRICS_PATH)

print(f"\nSaved re-run metrics: {RERUN_METRICS_PATH.resolve()}")

del model_best
del pred_best, pred_best_scaled, pred_best_raw
del y_test_true_best, y_test_pred_best
gc.collect()
_cleanup_torch_cache()


===== RE-RUN DLinear with BEST params (train+val -> test, split = 70/15/15) =====
Best params used:
  INPUT_CHUNK_LENGTH = 36
  OUTPUT_CHUNK_LENGTH = 24
  KERNEL_SIZE = 15
  N_EPOCHS = 30
  BATCH_SIZE = 128
  LR = 0.0001
  WEIGHT_DECAY = 1e-05
  CONST_INIT = True
  USE_REVIN = True
  USE_COVARIATES = False
  RANDOM_STATE = 42

===== BEST DLINEAR RESULTS (re-run: train+val -> test) =====
        Model Split     RMSE      MSE      MAE       R2
DLINEAR(best)  TEST 1.970394 3.882454 1.445427 0.925786

Saved re-run metrics: /home/jundian/Research-Project/project/best_dlinear_rerun_metrics.json


## autoFRK

In [16]:
# ============================================================
# 9) DLinear + autoFRK rolling residual forecast
#    Use DLinear(best) trained on train+val, then forecast test
# ============================================================
import gc
import numpy as np
import torch
from pathlib import Path
from autoFRK import AutoFRK
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("\n===== DLinear + autoFRK rolling residual forecast =====")

coords = coords_sample.astype(np.float64)
loc = torch.from_numpy(coords).to(dtype=torch.float64, device="cpu")

N = coords.shape[0]
H = T_test

mean_arr = mean_vec.values.astype(np.float64)
std_arr = std_vec.values.astype(np.float64)

if mean_arr.shape[0] != N:
    raise ValueError(f"mean_vec 長度 {mean_arr.shape[0]} 與格點數 N={N} 不一致")
if std_arr.shape[0] != N:
    raise ValueError(f"std_vec 長度 {std_arr.shape[0]} 與格點數 N={N} 不一致")
if ts_df.shape[1] != N:
    raise ValueError(f"ts_df 欄數 {ts_df.shape[1]} 與格點數 N={N} 不一致")
if trend_test_raw.shape != (H, N):
    raise ValueError(f"trend_test_raw shape {trend_test_raw.shape} 應為 {(H, N)}")

def inverse_scale_2d(x_TN):
    return x_TN * std_arr[None, :] + mean_arr[None, :]

def ts_to_TN(ts):
    arr = np.asarray(ts.all_values(copy=False))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr.astype(np.float64, copy=False)

input_chunk_length_best = int(bp["INPUT_CHUNK_LENGTH"])

# ------------------------------------------------------------
# 1) Build residual history on train+val
# ------------------------------------------------------------
y_hist_raw_TN = ts_df.iloc[:cut_val, :].to_numpy(dtype=np.float64)
hist_index = ts_df.index[:cut_val]
T_hist = y_hist_raw_TN.shape[0]

hist_forecast_kwargs = {
    "series": trainval_ts,
    "start": input_chunk_length_best,
    "forecast_horizon": 1,
    "stride": 1,
    "retrain": False,
    "last_points_only": True,
    "verbose": False,
}

if bool(bp.get("USE_COVARIATES", USE_COVARIATES)):
    hist_forecast_kwargs["past_covariates"] = month_trainval
    hist_forecast_kwargs["future_covariates"] = month_trainval

pred_hist = model_best.historical_forecasts(**hist_forecast_kwargs)

if isinstance(pred_hist, list):
    if len(pred_hist) == 0:
        raise RuntimeError("historical_forecasts 回傳空 list，無法建立 residual history。")
    pred_hist_ts = pred_hist[0]
    for k in range(1, len(pred_hist)):
        pred_hist_ts = pred_hist_ts.append(pred_hist[k])
else:
    pred_hist_ts = pred_hist

pred_hist_scaled_TN = ts_to_TN(pred_hist_ts)
pred_time_index = pred_hist_ts.time_index

trend_hat_hist_raw_TN = np.full((T_hist, N), np.nan, dtype=np.float64)

pos = hist_index.get_indexer(pred_time_index)
valid = pos >= 0
pred_hist_scaled_TN = pred_hist_scaled_TN[valid]
pos = pos[valid]

pred_hist_raw_TN = inverse_scale_2d(pred_hist_scaled_TN)
trend_hat_hist_raw_TN[pos] = pred_hist_raw_TN

R_hist_raw_TN = y_hist_raw_TN - trend_hat_hist_raw_TN
R_hist_raw_NT = R_hist_raw_TN.T

nan_ratio = float(np.isnan(R_hist_raw_NT).mean())
print("Initial residual history shape:", R_hist_raw_NT.shape, "| NaN ratio:", nan_ratio)

valid_cols = ~np.isnan(R_hist_raw_NT).any(axis=0)
R_hist_valid_NT = R_hist_raw_NT[:, valid_cols]

print("Residual history after dropping NaN columns:", R_hist_valid_NT.shape)

if R_hist_valid_NT.shape[1] < 2:
    raise RuntimeError("有效 residual 歷史長度不足，無法做 rolling latent forecast。")

# ------------------------------------------------------------
# 2) DLinear test trend
# ------------------------------------------------------------
trend_test_raw = pred_best_raw.astype(np.float64)
y_test_true_combo = test_true_raw.reshape(H, N).astype(np.float64)

# ------------------------------------------------------------
# 3) autoFRK rolling on DLinear residuals
# ------------------------------------------------------------
afrk = AutoFRK(dtype=torch.float64, device="cpu")

Rhat_roll_NH = np.zeros((N, H), dtype=np.float64)
K_list = []

R_hist_roll_NT = R_hist_valid_NT.copy()

for h in range(H):
    print(f"Rolling step {h + 1}/{H}", flush=True)

    data_hist = torch.from_numpy(R_hist_roll_NT).to(dtype=torch.float64)

    result_h = afrk.forward(
        data=data_hist,
        loc=loc,
        method="EM",
        maxit=50,
        tolerance=1e-6,
        n_neighbor=3,
        tps_method="spherical_fast",
    )

    w_hist = result_h["w"]
    w_hist_np = w_hist.detach().cpu().numpy()
    K, T_latent = w_hist_np.shape
    K_list.append(int(K))

    if T_latent < 2:
        raise RuntimeError(f"第 {h+1} 步 latent 歷史長度不足，無法建立 VAR(1)。")

    W0 = w_hist_np[:, :-1]
    W1 = w_hist_np[:, 1:]
    A = (W1 @ W0.T) @ np.linalg.pinv(W0 @ W0.T)

    w_last = w_hist_np[:, -1]
    w_next = (A @ w_last).reshape(K, 1)

    obj_next = dict(result_h)
    obj_next["w"] = torch.from_numpy(w_next).to(dtype=torch.float64)

    pred_res_h = afrk.predict(
        obj=obj_next,
        newloc=loc,
        se_report=False,
        tps_method="spherical_fast",
    )

    rhat_h = pred_res_h["pred.value"].detach().cpu().numpy()

    if rhat_h.ndim == 2 and rhat_h.shape == (N, 1):
        rhat_h = rhat_h[:, 0]
    elif rhat_h.ndim == 1 and rhat_h.shape[0] == N:
        pass
    else:
        raise ValueError(f"第 {h+1} 步 residual prediction shape 異常: {rhat_h.shape}")

    Rhat_roll_NH[:, h] = rhat_h

    y_true_h = y_test_true_combo[h, :]
    trend_h = trend_test_raw[h, :]
    r_true_h = y_true_h - trend_h

    R_hist_roll_NT = np.column_stack([R_hist_roll_NT, r_true_h])

# ------------------------------------------------------------
# 4) Final combined prediction and metrics
# ------------------------------------------------------------
Yhat_test_combo = trend_test_raw + Rhat_roll_NH.T

mse_combo = mean_squared_error(y_test_true_combo, Yhat_test_combo)
mae_combo = mean_absolute_error(y_test_true_combo, Yhat_test_combo)
rmse_combo = float(np.sqrt(mse_combo))
r2_combo = r2_score(y_test_true_combo, Yhat_test_combo)

mse_dlinear = mean_squared_error(y_test_true_combo, trend_test_raw)
mae_dlinear = mean_absolute_error(y_test_true_combo, trend_test_raw)
rmse_dlinear = float(np.sqrt(mse_dlinear))
r2_dlinear = r2_score(y_test_true_combo, trend_test_raw)

metrics_compare = pd.DataFrame({
    "Model": ["DLinear(best)", "DLinear + autoFRK rolling"],
    "Split": ["TEST", "TEST"],
    "RMSE": [rmse_dlinear, rmse_combo],
    "MSE": [float(mse_dlinear), float(mse_combo)],
    "MAE": [float(mae_dlinear), float(mae_combo)],
    "R2": [float(r2_dlinear), float(r2_combo)],
})

print("\n===== DLinear vs DLinear + autoFRK rolling =====")
print(metrics_compare.to_string(index=False))

FRK_METRICS_PATH = Path("dlinear_autofrk_rolling_test_metrics.json")
frk_payload = {
    "dlinear_best": {
        "RMSE": float(rmse_dlinear),
        "MSE": float(mse_dlinear),
        "MAE": float(mae_dlinear),
        "R2": float(r2_dlinear),
    },
    "dlinear_autofrk_rolling": {
        "RMSE": float(rmse_combo),
        "MSE": float(mse_combo),
        "MAE": float(mae_combo),
        "R2": float(r2_combo),
    },
    "N": int(N),
    "H": int(H),
    "K_first_step": int(K_list[0]) if len(K_list) > 0 else None,
    "K_last_step": int(K_list[-1]) if len(K_list) > 0 else None,
    "residual_history_nan_ratio": float(nan_ratio),
    "residual_history_length_before_drop": int(R_hist_raw_NT.shape[1]),
    "residual_history_length_after_drop": int(R_hist_valid_NT.shape[1]),
    "rerun_scheme": "DLinear train+val -> test, residual rolling on test",
    "split_ratio": {
        "train": 0.7,
        "val": 0.15,
        "test": 0.15,
    },
    "params_used": bp,
}
_json_dump(frk_payload, FRK_METRICS_PATH)
print(f"\nSaved rolling FRK metrics: {FRK_METRICS_PATH.resolve()}")

del afrk
gc.collect()
_cleanup_torch_cache()


===== DLinear + autoFRK rolling residual forecast =====


NameError: name 'trend_test_raw' is not defined